# LizyML Tutorial: Time Series Regression with LightGBM

This notebook demonstrates LizyML's time series CV workflow:

1. Data preparation (Jena Climate public dataset)
2. Config setup with `split.method: "time_series"`
3. Model fit (expanding-window CV with early stopping)
4. Evaluate — metrics table + split summary
5. Learning curve
6. Feature importance
7. Purged time series split (bonus)

**Dataset**: Jena Climate (10-minute intervals, sampled to hourly, ~70,000 rows)  
**Target**: `T (degC)` (temperature)  
**Features**: pressure, humidity, wind speed, etc.

## 1. Setup

In [ ]:
from __future__ import annotations

import pandas as pd

from lizyml import Model

## 2. Data Preparation

We use the **Jena Climate** dataset, a public weather time series recorded
at the Max Planck Institute for Biogeochemistry.  Original data is at
10-minute resolution; we resample to hourly for a manageable size.

In [ ]:
URL = (
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
    "jena_climate_2009_2016.csv.zip"
)

df_raw = pd.read_csv(URL)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Parse datetime and resample to hourly
df_raw["Date Time"] = pd.to_datetime(df_raw["Date Time"], format="%d.%m.%Y %H:%M:%S")
df_raw = df_raw.set_index("Date Time").resample("1h").mean().dropna().reset_index()

# Add integer time index for time series splits
df_raw["time_idx"] = range(len(df_raw))

print(f"Hourly shape: {df_raw.shape}")
print(f"Date range: {df_raw['Date Time'].min()} to {df_raw['Date Time'].max()}")
df_raw.head()

In [ ]:
# Use a subset (last 5000 rows) for faster tutorial execution
df = df_raw.tail(5000).reset_index(drop=True)
df["time_idx"] = range(len(df))

# Drop datetime column (not a feature)
df = df.drop(columns=["Date Time"])

print(f"Tutorial subset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.describe()

## 3. Config

Key differences from a standard regression config:

- `split.method: "time_series"` — expanding-window cross-validation (no future leakage)
- `data.time_col: "time_idx"` — enables `split_summary()` to show fold time ranges
- No `shuffle` or `random_state` — temporal order is preserved

In [ ]:
config = {
    "config_version": 1,
    "task": "regression",
    "data": {
        "target": "T (degC)",
        "time_col": "time_idx",
    },
    "split": {
        "method": "time_series",
        "n_splits": 5,
    },
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 6,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
        },
        "auto_num_leaves": True,
        "num_leaves_ratio": 0.8,
        "min_data_in_leaf_ratio": 0.01,
        "min_data_in_bin_ratio": 0.001,
    },
    "training": {
        "seed": 42,
        "early_stopping": {
            "enabled": True,
            "rounds": 50,
            "validation_ratio": 0.15,
        },
    },
    "evaluation": {
        "metrics": ["mae", "rmse", "r2"],
    },
}

## 4. Model Fit

In [ ]:
model = Model(config)
model.fit(data=df)
print("Fit complete.")

### 4.1 Resolved Parameters

In [ ]:
model.params_table()

## 5. Evaluate

### 5.1 Metrics Table

In [ ]:
model.evaluate_table().round(4)

### 5.2 Split Summary

`split_summary()` shows fold sizes and, when `data.time_col` is set,
the time range for each train/validation split.

In [ ]:
model.split_summary()

### 5.3 Learning Curve

In [ ]:
model.plot_learning_curve().show()

## 6. Feature Importance

In [ ]:
model.importance_plot(kind="split").show()

In [ ]:
imp = (
    pd.Series(model.importance(kind="gain"), name="importance_gain")
    .sort_values(ascending=False)
    .to_frame()
)
imp

## 7. Purged Time Series Split (Bonus)

`purged_time_series` adds a purge window around the train/validation boundary
to prevent target leakage from autocorrelated features.

In [ ]:
config_purged = {
    **config,
    "split": {
        "method": "purged_time_series",
        "n_splits": 3,
        "purge_gap": 24,  # 24 hours
        "embargo": 50,  # 50 observations
    },
}

model_purged = Model(config_purged)
model_purged.fit(data=df)
print("Purged fit complete.")

In [ ]:
model_purged.evaluate_table().round(4)

In [ ]:
model_purged.split_summary()

## 8. Predict on New Data

In [ ]:
# Use the last 100 rows as "new" data for prediction
df_new = df.drop(columns=["T (degC)"]).tail(100)
result = model.predict(df_new)
print(f"Predictions shape: {result.pred.shape}")
result.pred[:10]